In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from pyprojroot import here

In [3]:
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results" / "esda"
OUTPUT_DIR = here() / "results" / "validation"

In [4]:
sub_df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_subindex.csv", dtype={'CVEGEO': str})
relms_df = pd.read_csv(PROCESSED_DATA_DIR/"RELMS_index.csv", dtype={'CVEGEO': str})
helax_mun = "20014" # Ciudad Ixtepec, Oaxaca

In [5]:
sub_df.columns

Index(['CVEGEO', 'idx_resources', 'idx_energy', 'idx_logistics', 'idx_market',
       'idx_security'],
      dtype='object')

In [6]:
GENUINE_ZERO_CVEGEO_KNOWN = sub_df[sub_df['idx_market'] == 0]['CVEGEO'].astype(str).copy().tolist()

In [7]:
GENUINE_ZERO_CVEGEO_KNOWN

['20341',
 '7125',
 '19003',
 '5026',
 '20245',
 '26065',
 '26046',
 '20423',
 '20292',
 '8018',
 '20048',
 '8041',
 '24031',
 '5013',
 '24030',
 '20408',
 '20288',
 '28036',
 '32027']

In [8]:
idx_cols = [
    "idx_resources",
    "idx_energy",
    "idx_logistics",
    "idx_market",
    "idx_security",
]

In [9]:
sub_df["CVEGEO"] = sub_df["CVEGEO"].str.zfill(5)
relms_df["CVEGEO"] = relms_df["CVEGEO"].str.zfill(5)

In [10]:
X = sub_df[idx_cols].to_numpy(dtype=float)

### TOPSIS

In [30]:
norm_denom = np.sqrt((X ** 2).sum(axis=0))
X_norm = X / norm_denom
 
n_criteria = X.shape[1]
weights = np.full(n_criteria, 1.0 / n_criteria)
X_weighted = X_norm * weights
 
ideal_best = X_weighted.max(axis=0)
ideal_worst = X_weighted.min(axis=0)
 
dist_best = np.sqrt(((X_weighted - ideal_best) ** 2).sum(axis=1))
dist_worst = np.sqrt(((X_weighted - ideal_worst) ** 2).sum(axis=1))
 
closeness = dist_worst / (dist_best + dist_worst)
 
sub_df["TOPSIS_score"] = closeness
sub_df["rank_topsis"] = sub_df["TOPSIS_score"].rank(ascending=False, method="min").astype(int)
sub_df.to_csv(OUTPUT_DIR/"topsis_vs_relms.csv")

In [12]:
relms_df = relms_df[["CVEGEO", "RELMS_index"]]
relms_df["rank_relms"] = relms_df["RELMS_index"].rank(ascending=False, method="min").astype(int)
 
df = relms_df.merge(
    sub_df[["CVEGEO", "TOPSIS_score", "rank_topsis"]], on="CVEGEO", how="inner"
)

### Spearman rank correlation: RELMS vs TOPSIS

In [13]:
rho, p_val = spearmanr(df["RELMS_index"], df["TOPSIS_score"])
 
print("=" * 60)
print("SPEARMAN RANK CORRELATION (RELMS vs TOPSIS)")
print("=" * 60)
print(f"  rho = {rho:.4f}, p-value = {p_val:.4e}")
print()

SPEARMAN RANK CORRELATION (RELMS vs TOPSIS)
  rho = 0.8839, p-value = 0.0000e+00



### TOPSIS vs RELMS overlaps

In [29]:
keys = [10, 20, int(np.ceil(len(df) * 0.05))]
top_dict = pd.DataFrame()
for top_n in keys:
    top_relms = set(df.nsmallest(top_n, "rank_relms")["CVEGEO"])
    top_topsis = set(df.nsmallest(top_n, "rank_topsis")["CVEGEO"])
    olp = top_relms & top_topsis
    overlap = len(olp)
    
    print(f"Top {top_n}: overlap = {overlap}/{top_n} "
          f"({overlap / top_n * 100:.1f}%)")
    print(f"Overlaping municipalities:")
    print(olp)
print()

Top 10: overlap = 3/10 (30.0%)
Overlaping municipalities:
{'29005', '20557', '29043'}
Top 20: overlap = 9/20 (45.0%)
Overlaping municipalities:
{'20557', '20087', '31101', '31050', '20014', '29043', '29048', '20441', '29005'}
Top 124: overlap = 50/124 (40.3%)
Overlaping municipalities:
{'07012', '29059', '20305', '20557', '29010', '29018', '29031', '30105', '31050', '31041', '20014', '16107', '20441', '29005', '30059', '20043', '20079', '20030', '29041', '29015', '20087', '29026', '07040', '31102', '13010', '20124', '20157', '30123', '29028', '16105', '20409', '19006', '31101', '30048', '30207', '30026', '19019', '29043', '29002', '29001', '29060', '20553', '15010', '29039', '07071', '19046', '29053', '29048', '13013', '29038'}



### RELMS vs TOPSIS zero municipalities

In [14]:
print("=" * 60)
print("GENUINE-ZERO MUNICIPALITIES: RELMS vs TOPSIS RANK")
print("=" * 60)
zero_check = df[df["CVEGEO"].isin(GENUINE_ZERO_CVEGEO_KNOWN)]
print(
    zero_check[["CVEGEO", "RELMS_index", "rank_relms", "TOPSIS_score", "rank_topsis"]]
    .round(4)
    .to_string(index=False)
)


GENUINE-ZERO MUNICIPALITIES: RELMS vs TOPSIS RANK
CVEGEO  RELMS_index  rank_relms  TOPSIS_score  rank_topsis
 20341       0.0062        2460        0.3958         1884
 19003       0.0061        2462        0.3896         1966
 20245       0.0059        2464        0.3727         2190
 26065       0.0059        2465        0.3631         2285
 26046       0.0059        2466        0.3629         2290
 20423       0.0058        2467        0.3672         2239
 20292       0.0056        2468        0.3829         2072
 20048       0.0055        2470        0.3693         2224
 24031       0.0054        2472        0.3251         2460
 24030       0.0053        2474        0.3197         2466
 20408       0.0053        2475        0.3406         2419
 20288       0.0052        2476        0.3453         2397
 28036       0.0052        2477        0.3240         2462
 32027       0.0052        2478        0.3384         2423


### RELMS vs TOPSIS helax municipality

In [15]:
helax_row = df[df["CVEGEO"] == helax_mun]
print("=" * 60)
print("HELAX MUNICIPALITY: RELMS vs TOPSIS RANK")
print("=" * 60)
print(
    helax_row[["CVEGEO", "RELMS_index", "rank_relms", "TOPSIS_score", "rank_topsis"]]
    .round(4)
    .to_string(index=False)
)

HELAX MUNICIPALITY: RELMS vs TOPSIS RANK
CVEGEO  RELMS_index  rank_relms  TOPSIS_score  rank_topsis
 20014       0.6468           4        0.6464           14


In [16]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(df["TOPSIS_score"], df["RELMS_index"], s=15, alpha=0.4, color="steelblue")
 

ax.scatter(
    helax_row["TOPSIS_score"],
    helax_row["RELMS_index"],
    s=80,
    color="crimson",
    marker="*",
    label="Helax municipality",
    zorder=5,
)
ax.legend()
 
ax.set_xlabel("TOPSIS Closeness Coefficient")
ax.set_ylabel("RELMS Index (geometric mean)")
#ax.set_title("RELMS vs TOPSIS — Methodological Triangulation")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "relms_vs_topsis_scatter.png", dpi=300)
plt.close(fig)
 
print(f"Plots saved to: {OUTPUT_DIR.resolve()}")

Plots saved to: C:\Users\franc\OneDrive\Documents\Repositories\hydrogen-hub-index\results\validation
